In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

import matplotlib.pyplot as plt



df = pd.read_csv("BankChurners.csv")

# Remove unnecessary columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]



df['Attrition_Flag'] = df['Attrition_Flag'].map({
    'Existing Customer': 0,
    'Attrited Customer': 1
})



le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])



X = df.drop('Attrition_Flag', axis=1)
y = df['Attrition_Flag']



scaler = StandardScaler()
X = scaler.fit_transform(X)



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [ ]:
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Layer 1
model.add(Dense(
    16,
    activation='relu',
    input_shape=(X_train.shape[1],),
    kernel_regularizer=l2(0.001)
))
model.add(BatchNormalization())
model.add(Dropout(0.9))

# Layer 2
model.add(Dense(
    64,
    activation='relu',
    kernel_regularizer=l2(0.001)
))
model.add(BatchNormalization())
model.add(Dropout(0.8))

# Output Layer
model.add(Dense(1, activation='sigmoid'))


model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [125]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5636 - loss: 1.1075 - val_accuracy: 0.8365 - val_loss: 0.5142
Epoch 2/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6610 - loss: 0.8354 - val_accuracy: 0.8378 - val_loss: 0.4410
Epoch 3/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7375 - loss: 0.6553 - val_accuracy: 0.8371 - val_loss: 0.4075
Epoch 4/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7701 - loss: 0.6031 - val_accuracy: 0.8371 - val_loss: 0.3801
Epoch 5/5
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8079 - loss: 0.5250 - val_accuracy: 0.8371 - val_loss: 0.3580


: 